In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists

import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')

HEADER_FILE='../headers_auth.json'
PLAYCOUNT_FILE='../playlists/_ytmusic_lastfm_match_id_map.tsv'
NOT_LIKE_FILE  = '../playlists/_not_liked_tracks.tsv'
Y = YTMusicPlaylists(header=HEADER_FILE, playcount_map=PLAYCOUNT_FILE, not_like_tsv=NOT_LIKE_FILE)
print(Y.playlists['title'].unique())

Using ytmusicapi version: 0.24.1
Using header file: ../headers_auth.json
Loaded 266580 playounts from 104345 tracks
['Your Likes' 'zz_mixes_traktor' 'zz_mixes library' 'zz_dj_remix'
 'zz_dj_history_2015y04m11d_02h18m16s' 'zz_dj_gold'
 'zz_dj_fx + drumz + vox' 'zz_dj_edits'
 'zz_ Live concert phone recordings' 'z_dj_thumbs_up' 'z__thumbs_up'
 'yz_2019_unrated_albums' 'yz_2018_unrated_albums'
 'yz_2017_unrated_albums' 'yz__seed_albums' 'yz__ipod_albums'
 'yyz_musicbee_library_albums_part_9' 'yyz_musicbee_library_albums_part_8'
 'yyz_musicbee_library_albums_part_7' 'yyz_musicbee_library_albums_part_6'
 'yyz_musicbee_library_albums_part_5' 'yyz_musicbee_library_albums_part_4'
 'yyz_musicbee_library_albums_part_3' 'yyz_musicbee_library_albums_part_2'
 'yyz_musicbee_library_albums_part_12'
 'yyz_musicbee_library_albums_part_11'
 'yyz_musicbee_library_albums_part_10'
 'yyz_musicbee_library_albums_part_1'
 'yyz_musicbee_download_albums_part_1'
 'y_2022_top_70_unrated_albums_12-29-2022' 'y_2022

# Process Each Playlist

##### TODO move based on playcount (if not LIKE infer NOT_LIKE based on large playcount)
##### TODO for NOT_LIKE?
##### TODO make into script, run monthly



In [15]:
playlists_kinds = {k: set() for k in Y._valid_playlist_kinds}
for i, p in Y.playlists.iterrows():
  print(p.title)

Your Likes
zz_mixes_traktor
zz_mixes library
zz_dj_remix
zz_dj_history_2015y04m11d_02h18m16s
zz_dj_gold
zz_dj_fx + drumz + vox
zz_dj_edits
zz_ Live concert phone recordings
z_dj_thumbs_up
z__thumbs_up
yz_2019_unrated_albums
yz_2018_unrated_albums
yz_2017_unrated_albums
yz__seed_albums
yz__ipod_albums
yyz_musicbee_library_albums_part_9
yyz_musicbee_library_albums_part_8
yyz_musicbee_library_albums_part_7
yyz_musicbee_library_albums_part_6
yyz_musicbee_library_albums_part_5
yyz_musicbee_library_albums_part_4
yyz_musicbee_library_albums_part_3
yyz_musicbee_library_albums_part_2
yyz_musicbee_library_albums_part_12
yyz_musicbee_library_albums_part_11
yyz_musicbee_library_albums_part_10
yyz_musicbee_library_albums_part_1
yyz_musicbee_download_albums_part_1
y_2022_top_70_unrated_albums_12-29-2022
y_2022_thumbs_up
y_2022_musicbee_albums
y_2021_top_50_collage_albums_12-12-2022
y_2021_thumbs_up
y_2021_musicbee_albums
y_2020_top_50_albums_04-05-2021
y_2020_top_100_unrated_albums_05-19-2021
y_2020

In [16]:
# Skip playlistes inferred as these kind
DRY_RUN = False  # make sure all NOT OK is fine

VPRINT = False  # Verbose printing
WARN_PRINT = True  # Print warnings

# Options for checking inferred playlist kind
SKIP_PLAYLIST_KINDS = ('SKIP', 'ALBUM', 'YT_GENERATED')
LIKE_MIN_LIKE_PCT = 80
NOTLIKE_MAX_LIKE_PCT = 20
RADIO_MAX_LIKE_PCT = 50

# Options for processing playlists
MIN_RADIO_LIKE_TO_SPLIT = 10
DUPLICATE_THRESHOLD = 3  # was 4 first run

playlists_kinds = {k: set() for k in Y._valid_playlist_kinds}
for i, p in Y.playlists.iterrows():
    if p.title.startswith('zz not like'):
        continue

    if VPRINT:
        print(100*'=' + f'\nPlaylist: {p.title} ({p.playlistId})',
              f'has {p.count} tracks')

    """Potentially skip playlist"""
    # Infer playlist kind from the title, default to LIKE if nothing inferred
    pl_kind = Y.infer_playlist_kind(p)
    if not pl_kind:
        pl_kind = 'LIKE'

    # Decide to skip playlist based on playlist kind
    playlists_kinds[pl_kind].add(p.title)
    if pl_kind in SKIP_PLAYLIST_KINDS:
        print(f'SKIPPING playlist: {p.title} as it is',
              f'a kind flagged for skipping: {pl_kind}')
        continue

    """Query playlist tracks then potentially skip"""
    # Query playlist tracks and other metadata
    p_info = Y.playlist_get_info(
        p.playlistId, playlist_limit=Y.playlist_limit).copy()
    if p_info['trackCount'] == 0:
        print('No tracks in playlist')
        continue
    # Check max length of playlist
    if len(p_info['tracks']) >= Y.playlist_limit:
        print(f'SKIPPING playlist: {p.title} which has',
              f'{Y.playlist_limit} or more tracks ({len(p_info["tracks"])})')
        continue

    # Check playlist privacy
    if p_info['privacy'] == 'PUBLIC':
        print(f'SKIPPING playlist: {p.title} which has',
              f'privacy: {p_info["privacy"]}')
        continue
    elif p_info['privacy'] == 'UNLISTED' and WARN_PRINT:
        print(f'WARNING {p_info["privacy"]} playlist: {p.title}')

    # Get ratings for playlist tracks
    ratings = {k: set() for k in Y._valid_ratings}
    for track in p_info["tracks"]:
        if track["likeStatus"] not in ratings.keys():
            ratings['NONE'].add(track["videoId"])
        else:
            ratings[track["likeStatus"]].add(track["videoId"])

    # See if playlist is correctly flagged as LIKE or RADIO
    like_percent = round(100*len(ratings["LIKE"])/len(p_info["tracks"]))
    if not Y._is_playlist_kind_ok(pl_kind, like_percent,
                                  LIKE_MIN_LIKE_PCT, NOTLIKE_MAX_LIKE_PCT,
                                  RADIO_MAX_LIKE_PCT):
        if WARN_PRINT:
            print(f'WARNING NOT OK {pl_kind} Playlist: {p.title}',
                  f'({like_percent}% liked) has: {len(ratings["LIKE"])} likes,',
                  f'{len(ratings["DISLIKE"])} dislikes,{len(ratings["INDIFFERENT"])}',
                  f'indifferent, {len(ratings["NONE"])} none')

    """Potentially alter playlist, or generate new playlists"""
    if DRY_RUN:
        continue

    # Remove duplicates from playlist
    new_pl_id = Y.playlist_remove_duplicates(
        p_info, duplicate_threshold=DUPLICATE_THRESHOLD, verbose=VPRINT)
    if p.playlistId != new_pl_id:
        p.playlistId = new_pl_id
        p_info = Y.playlist_get_info(
            new_pl_id, playlist_limit=Y.playlist_limit, use_cache=False)

    # Like all tracks in playlist if kind is LIKE
    if pl_kind == 'LIKE':
        Y.playlist_rate_all_songs(
            p_info, rating=pl_kind, skip_if_dislike=True, verbose=VPRINT)
        continue
    # Split radio playlist into LIKE vs RADIO
    elif pl_kind == 'INDIFFERENT':
        Y.clean_up_radio_playlist(
            p_info, verbose=VPRINT,  move_like=True, 
            min_num_like=MIN_RADIO_LIKE_TO_SPLIT,
            create_like_playlist=True,
            remove_dislike=True, remove_not_like=True
        )
        continue


Playlist folk 1960s: Rated 0 of 99 tracks as LIKE
Playlist Folk: Rated 0 of 200 tracks as LIKE
Playlist electronic witch_house: Rated 3 of 56 tracks as LIKE
Playlist electronic we are alone: Rated 0 of 49 tracks as LIKE
Playlist electronic uk: Rated 9 of 264 tracks as LIKE
Playlist electronic trance dj: Rated 2 of 27 tracks as LIKE
Not splitting playlist electronic soft pad radio,  not enough likes (0)
Playlist electronic soft pad: Rated 1 of 55 tracks as LIKE
Not splitting playlist electronic radio,  not enough likes (4)
Playlist electronic new indie beats: Rated 6 of 405 tracks as LIKE
Playlist electronic jaar: Rated 1 of 38 tracks as LIKE
Not splitting playlist electronic Innerwaves radio,  not enough likes (2)
Playlist electronic Innerwaves: Rated 0 of 32 tracks as LIKE
Not splitting playlist electronic indie radio,  not enough likes (1)
Playlist electronic indie essentials: Rated 0 of 52 tracks as LIKE
Not splitting playlist electronic House Special radio,  not enough likes (1)
Pl